In [1]:
# Cell 1: imports, reproducibility, device, constants
import os, json, math, random, time, traceback
from pathlib import Path
from collections import Counter
from typing import List, Dict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from tqdm import tqdm

# torch_geometric
try:
    import torch_geometric
    from torch_geometric.data import Data, Batch
    from torch_geometric.utils import to_dense_batch
except Exception as e:
    torch_geometric = None
    print("Warning: torch_geometric import failed. Install torch_geometric matching your torch/CUDA. Error:", e)

# reproducibility
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "torch:", torch.__version__)
if torch_geometric is not None:
    try: print("torch_geometric:", torch_geometric.__version__)
    except: pass

# Paths (edit if needed)
DATA_DIR = Path(".")
VATEX_FEAT_DIR = Path("./FEAT_DIR")   # point this to the directory with .npy features
TRAIN_JSON = DATA_DIR / "vatex_training_v1.0.json"
VAL_JSON   = DATA_DIR / "vatex_validation_v1.0.json"

# Base constants (you can tune later)
TEXT_EMB_SIZE   = 256
VIDEO_EMB_SIZE  = 1024
GNN_HIDDEN_SIZE = 512
MAX_TOK_LEN = 40
MAX_VID_SEG = 64

print("Constants:", TEXT_EMB_SIZE, VIDEO_EMB_SIZE, GNN_HIDDEN_SIZE, MAX_TOK_LEN, MAX_VID_SEG)
print("Files exist? TRAIN_JSON:", TRAIN_JSON.exists(), "FEAT_DIR:", VATEX_FEAT_DIR.exists())

Device: cuda torch: 2.8.0+cu126
torch_geometric: 2.6.1
Constants: 256 1024 512 40 64
Files exist? TRAIN_JSON: True FEAT_DIR: True


In [2]:
# Cell 2: load VATEX annotations and build EN-ZH pairs (parallel captions indices 5..9)
def load_json(path: Path):
    if not path.exists():
        print(f"Warning: JSON not found at {path}")
        return []
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_ann = load_json(TRAIN_JSON)
val_ann   = load_json(VAL_JSON)
print("Train ann:", len(train_ann), "Val ann:", len(val_ann))

def build_pairs(ann_list):
    pairs=[]
    for item in ann_list:
        vid = item.get("videoID")
        en = item.get("enCap", [])
        ch = item.get("chCap", [])
        n = min(len(en), len(ch))
        # indices 5..9 if available
        for i in range(5, min(10, n)):
            pairs.append({"video_id": vid, "en": en[i], "zh": ch[i]})
    return pairs

train_pairs = build_pairs(train_ann)
val_pairs   = build_pairs(val_ann)
print("Train pairs:", len(train_pairs), "Val pairs:", len(val_pairs))
if len(train_pairs)>0: print("Sample pair:", train_pairs[0])


Train ann: 25991 Val ann: 3000
Train pairs: 129955 Val pairs: 15000
Sample pair: {'video_id': 'Ptf_2VRj-V0_000122_000132', 'en': 'a man going down a rock cliff using a harness and rope with another person nearby', 'zh': '一个人借助绳子从山上爬了下来走到另外一个人身边搭着背说着话。'}


In [3]:
# Cell 3: load source tokenizer (mBART) & build char-level target vocabulary
SRC_TOK_NAME = "facebook/mbart-large-50"

tok_src = AutoTokenizer.from_pretrained(SRC_TOK_NAME, src_lang="en_XX", use_fast=True)
print("Loaded source tokenizer:", SRC_TOK_NAME, "vocab_size:", tok_src.vocab_size)

# Build char-level Chinese vocabulary from training data
chars = Counter()
for item in train_ann:
    for s in item.get("chCap", []):
        chars.update(list(s.strip()))

special_tokens = ["<pad>", "<unk>", "<bos>", "<eos>"]
char_list = special_tokens + sorted(chars.keys())
char2id = {c:i for i,c in enumerate(char_list)}
id2char = {i:c for i,c in enumerate(char_list)}

def char_encode(s, max_len=MAX_TOK_LEN):
    ids = [char2id["<bos>"]]
    for ch in list(s)[:max_len-2]:
        ids.append(char2id.get(ch, char2id["<unk>"]))
    ids.append(char2id["<eos>"])
    return ids

def char_decode(ids):
    return "".join([id2char[i] for i in ids if i >= len(special_tokens)])

PAD_IDX = char2id["<pad>"]
BOS_IDX = char2id["<bos>"]
EOS_IDX = char2id["<eos>"]
SRC_VOCAB_SIZE = tok_src.vocab_size
TGT_VOCAB_SIZE = len(char_list)

print("Built char-level target vocab size:", TGT_VOCAB_SIZE)
if len(val_pairs)>0:
    s = val_pairs[0]["zh"]
    print("Validation example (zh):", s)
    print("char_encode:", char_encode(s)[:50])
    print("char_decode:", char_decode(char_encode(s)[:50]))


Loaded source tokenizer: facebook/mbart-large-50 vocab_size: 250054
Built char-level target vocab size: 3655
Validation example (zh): 一个人在借助绳子的条件下顺利的从山上爬下来。
char_encode: [2, 95, 120, 181, 750, 281, 433, 2572, 910, 2246, 1631, 206, 102, 3497, 398, 2246, 191, 1005, 101, 2077, 102, 1632, 92, 3]
char_decode: 一个人在借助绳子的条件下顺利的从山上爬下来。


In [4]:
# Cell 4: feature loader + Dataset using char-level targets + collate function
def ensure_video_features(video_id: str) -> torch.Tensor:
    subfolders = [
        VATEX_FEAT_DIR / "trainval" / "train",
        VATEX_FEAT_DIR / "trainval" / "val",
        VATEX_FEAT_DIR / "public_test" / "public_test",
        VATEX_FEAT_DIR / "private_test" / "private_test"
    ]
    for folder in subfolders:
        path = folder / f"{video_id}.npy"
        if path.exists():
            arr = np.load(path)
            if arr.ndim == 3 and arr.shape[0] == 1:
                arr = arr.squeeze(0)
            if arr.shape[0] > MAX_VID_SEG:
                arr = arr[:MAX_VID_SEG]
            feats = torch.from_numpy(arr).float()
            if feats.dim()==1:
                feats = feats.unsqueeze(0)
            D = feats.size(-1)
            if D != VIDEO_EMB_SIZE:
                if D < VIDEO_EMB_SIZE:
                    pad = torch.zeros((feats.size(0), VIDEO_EMB_SIZE - D))
                    feats = torch.cat([feats, pad], dim=-1)
                else:
                    feats = feats[:, :VIDEO_EMB_SIZE]
            feats = F.normalize(feats, dim=-1, eps=1e-6)
            feats = torch.nan_to_num(feats, nan=0.0, posinf=1e4, neginf=-1e4)
            return feats
    return torch.zeros((1, VIDEO_EMB_SIZE), dtype=torch.float32)

class GraphVATEXDataset(Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        p = self.pairs[idx]
        en_ids = tok_src.encode(p["en"], add_special_tokens=True, truncation=True, max_length=MAX_TOK_LEN)
        zh_ids = char_encode(p["zh"], max_len=MAX_TOK_LEN)
        if len(en_ids)==0: en_ids = [tok_src.unk_token_id or 3]
        if len(zh_ids)==0: zh_ids = [char2id.get("<unk>",1)]
        feats = ensure_video_features(p["video_id"])
        return {
            "video_id": p["video_id"],
            "src_tokens": torch.tensor(en_ids, dtype=torch.long),
            "tgt_tokens": torch.tensor(zh_ids, dtype=torch.long),
            "video_features": feats
        }

def collate_graph_vatex(batch):
    srcs=[b["src_tokens"] for b in batch]
    tgts=[b["tgt_tokens"] for b in batch]
    feats=[b["video_features"] for b in batch]
    src_pad=nn.utils.rnn.pad_sequence(srcs, padding_value=tok_src.pad_token_id, batch_first=True)
    tgt_pad=nn.utils.rnn.pad_sequence(tgts, padding_value=PAD_IDX, batch_first=True)
    return {
        "src_tokens": src_pad.to(DEVICE),
        "tgt_tokens": tgt_pad.to(DEVICE),
        "src_padding_mask": (src_pad==tok_src.pad_token_id).to(DEVICE),
        "tgt_padding_mask": (tgt_pad==PAD_IDX).to(DEVICE),
        "video_features_list": feats
    }

# DataLoaders (start with small batch sizes)
BATCH_SIZE = 4
train_dl = DataLoader(GraphVATEXDataset(train_pairs), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_graph_vatex)
val_dl   = DataLoader(GraphVATEXDataset(val_pairs), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_graph_vatex)
print("Train batches:", len(train_dl), "Val batches:", len(val_dl))

# quick batch inspect
batch = next(iter(train_dl))
print("src shape:", batch["src_tokens"].shape, "tgt shape:", batch["tgt_tokens"].shape)
for i,vf in enumerate(batch["video_features_list"]):
    print(f"video_features[{i}] shape:", vf.shape)
print("sample tgt ids:", batch["tgt_tokens"][0,:20].tolist())
print("decoded sample (char):", char_decode([int(x) for x in batch["tgt_tokens"][0,:20].tolist() if int(x) < TGT_VOCAB_SIZE]))


Train batches: 32489 Val batches: 3750
src shape: torch.Size([4, 31]) tgt shape: torch.Size([4, 23])
video_features[0] shape: torch.Size([32, 1024])
video_features[1] shape: torch.Size([32, 1024])
video_features[2] shape: torch.Size([32, 1024])
video_features[3] shape: torch.Size([32, 1024])
sample tgt ids: [2, 95, 120, 2196, 922, 1773, 750, 1001, 2360, 195, 1274, 3310, 1941, 1256, 1616, 3310, 1790, 95, 120, 1368]
decoded sample (char): 一个男孩正在展示他手里游戏机里每一个按


In [5]:
# Cell 5: PositionalEncoding, TokenEmbedding, mask, graph builder (with eps and nan guards)
class PositionalEncoding(nn.Module):
    def __init__(self, emb_size, dropout=0.1, maxlen=5000):
        super().__init__()
        den = torch.exp(-torch.arange(0, emb_size, 2, dtype=torch.float32) * math.log(10000.0) / emb_size)
        pos = torch.arange(0, maxlen, dtype=torch.float32).reshape(maxlen, 1)
        pos_emb = torch.zeros((maxlen, emb_size), dtype=torch.float32)
        pos_emb[:, 0::2] = torch.sin(pos * den)
        pos_emb[:, 1::2] = torch.cos(pos * den)
        pos_emb = pos_emb.unsqueeze(0)
        self.register_buffer("pos_emb", pos_emb)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        x = x + self.pos_emb[:, :x.size(1), :].to(x.dtype)
        return self.dropout(x)

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, emb_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_size)
        self.emb_size = emb_size
    def forward(self, tokens):
        return self.emb(tokens) * math.sqrt(self.emb_size)

def generate_square_subsequent_mask(sz, device):
    return torch.triu(torch.ones((sz, sz), device=device), 1).bool()

def build_text_video_graph(text_feats, video_feats, k=3):
    Lt, Tv = text_feats.size(0), video_feats.size(0)
    x = torch.cat([text_feats, video_feats], 0)
    edges=[]
    for i in range(max(0,Lt-1)):
        edges += [[i, i+1], [i+1, i]]
    off = Lt
    for i in range(max(0, Tv-1)):
        edges += [[off+i, off+i+1], [off+i+1, off+i]]
    if Tv>0 and Lt>0:
        with torch.no_grad():
            t = F.normalize(text_feats, dim=-1, eps=1e-6)
            v = F.normalize(video_feats, dim=-1, eps=1e-6)
            t = torch.nan_to_num(t, nan=0.0, posinf=1e4, neginf=-1e4)
            v = torch.nan_to_num(v, nan=0.0, posinf=1e4, neginf=-1e4)
            sim = t @ v.t()
            if not torch.isfinite(sim).all():
                sim = torch.nan_to_num(sim, nan=0.0, posinf=1e4, neginf=-1e4)
            topv = torch.topk(sim, k=min(k, Tv), dim=1).indices
            for i in range(Lt):
                for j in topv[i].tolist():
                    edges += [[i, off+j], [off+j, i]]
    if len(edges)==0:
        edges = [[0,0]]
    edge_index = torch.tensor(edges, dtype=torch.long).T.contiguous()
    return Data(x=x, edge_index=edge_index)


In [6]:
# Cell 6: Model definition (requires torch_geometric)
if torch_geometric is None:
    raise RuntimeError("torch_geometric required. Install matching version for your torch/CUDA.")

class Seq2SeqTransformerGraphVideo(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, video_dim=VIDEO_EMB_SIZE, text_dim=TEXT_EMB_SIZE,
                 hidden=GNN_HIDDEN_SIZE, nhead=8, num_dec_layers=3, dropout=0.1):
        super().__init__()
        self.src_tok = TokenEmbedding(src_vocab, text_dim)
        self.tgt_tok = TokenEmbedding(tgt_vocab, text_dim)
        self.pos_enc = PositionalEncoding(text_dim, dropout)

        self.text_proj = nn.Linear(text_dim, hidden)
        self.video_proj = nn.Linear(video_dim, hidden)
        self.text_norm = nn.LayerNorm(hidden)
        self.video_norm = nn.LayerNorm(hidden)

        self.type_emb = nn.Embedding(2, hidden)
        self.pre_norm = nn.LayerNorm(hidden)
        self.pre_drop = nn.Dropout(0.1)

        self.gnn1 = torch_geometric.nn.GATv2Conv(hidden, hidden, heads=nhead, dropout=dropout, add_self_loops=True)
        self.gnn1n = nn.LayerNorm(hidden * nhead)
        self.gnn2 = torch_geometric.nn.GATv2Conv(hidden * nhead, hidden, heads=1, dropout=dropout, add_self_loops=True)
        self.gnn2n = nn.LayerNorm(hidden)

        self.dec_proj = nn.Linear(text_dim, hidden)
        dec_layer = nn.TransformerDecoderLayer(d_model=hidden, nhead=nhead, dropout=dropout, batch_first=True)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_dec_layers)

        self.generator = nn.Linear(hidden, tgt_vocab)
        nn.init.zeros_(self.generator.bias)

    def _encode_graph_batch(self, src_tokens, src_mask, vfeats_list):
        B, T = src_tokens.size()
        src_emb = self.pos_enc(self.src_tok(src_tokens))
        src_proj = self.text_norm(self.text_proj(src_emb))
        data_list=[]
        for i in range(B):
            Lt = int((~src_mask[i]).sum().item())
            Lt = max(Lt, 1)
            t = src_proj[i, :Lt, :]
            v_raw = vfeats_list[i].to(src_tokens.device)
            v = self.video_norm(self.video_proj(v_raw))
            g = build_text_video_graph(t, v, k=3)
            # ensure node_type constructed on same device
            g.node_type = torch.cat([torch.zeros(t.size(0)), torch.ones(v.size(0))]).long().to(src_tokens.device)
            data_list.append(g)

        graph_batch = Batch.from_data_list(data_list).to(src_tokens.device)
        x = graph_batch.x
        ei = graph_batch.edge_index
        # device/dtype guards
        if ei.dtype != torch.long: ei = ei.long()
        ei = ei.to(x.device)
        types = graph_batch.node_type.to(x.device)

        x = self.pre_drop(self.pre_norm(x)) + self.type_emb(types.to(x.device))
        x = torch.nan_to_num(x, nan=0.0, posinf=1e4, neginf=-1e4)

        x = self.gnn1(x, ei)
        x = torch.nan_to_num(x, nan=0.0, posinf=1e4, neginf=-1e4)
        x = F.relu(self.gnn1n(x))

        x = self.gnn2(x, ei)
        x = torch.nan_to_num(x, nan=0.0, posinf=1e4, neginf=-1e4)
        x = self.gnn2n(x)

        mem, valid_mask = to_dense_batch(x, batch=graph_batch.batch)
        mem_keypad_mask = ~valid_mask
        return mem, mem_keypad_mask

    def forward(self, batch):
        src, tgt = batch["src_tokens"], batch["tgt_tokens"]
        smask, tmask = batch["src_padding_mask"], batch["tgt_padding_mask"]
        vlist = batch["video_features_list"]

        tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
        tgt_pad = tmask[:, :-1]

        mem, mem_mask = self._encode_graph_batch(src, smask, vlist)

        tgt_emb = self.pos_enc(self.tgt_tok(tgt_in))
        tgt_proj = self.dec_proj(tgt_emb)
        tgt_causal = generate_square_subsequent_mask(tgt_in.size(1), tgt_in.device)

        dec_out = self.decoder(tgt_proj, mem,
                               tgt_mask=tgt_causal,
                               tgt_key_padding_mask=tgt_pad,
                               memory_key_padding_mask=mem_mask)
        logits = self.generator(dec_out)
        return logits, tgt_out


In [7]:
# Cell 7: helpers - move batch to device, translate_one_safe, translate_beam
def move_batch_to_device(batch, device):
    batch2 = {}
    for k,v in batch.items():
        if k == "video_features_list":
            batch2[k] = [vf.to(device) for vf in v]
        else:
            batch2[k] = v.to(device)
    return batch2

@torch.no_grad()
def translate_one_safe(model, item, max_len=64, min_len=5, rep_pen=5.0):
    model.eval()
    batch = collate_graph_vatex([item])
    memory, mem_mask = model._encode_graph_batch(batch["src_tokens"], batch["src_padding_mask"], batch["video_features_list"])
    ys = torch.tensor([[BOS_IDX]], device=DEVICE)
    for _ in range(max_len - 1):
        tmask = generate_square_subsequent_mask(ys.size(1), DEVICE)
        tgt_emb = model.pos_enc(model.tgt_tok(ys))
        tgt_proj = model.dec_proj(tgt_emb)
        out = model.decoder(tgt_proj, memory, tgt_mask=tmask, memory_key_padding_mask=mem_mask)
        logits = model.generator(out[:, -1, :])
        log_probs = F.log_softmax(logits, dim=-1)
        used_tokens = set(ys.flatten().tolist())
        log_probs = log_probs.clone()
        for tid in used_tokens:
            if 0 <= tid < log_probs.size(1):
                log_probs[0, tid] -= rep_pen
        if ys.size(1) < min_len and 0 <= EOS_IDX < log_probs.size(1):
            log_probs[0, EOS_IDX] = -1e9
        next_id = int(torch.argmax(log_probs, -1))
        ys = torch.cat([ys, torch.tensor([[next_id]], device=DEVICE)], 1)
        if next_id == EOS_IDX:
            break
    ids = [i for i in ys.flatten().tolist() if i not in (BOS_IDX, EOS_IDX, PAD_IDX)]
    return char_decode(ids)

@torch.no_grad()
def translate_beam(model, item, beam_size=4, max_len=64, length_penalty=0.7, rep_pen=3.0):
    model.eval()
    batch = collate_graph_vatex([item])
    memory, mem_mask = model._encode_graph_batch(batch["src_tokens"], batch["src_padding_mask"], batch["video_features_list"])
    device = DEVICE
    beams = [(torch.tensor([[BOS_IDX]], device=device), 0.0)]
    completed = []
    for step in range(max_len):
        new_beams = []
        for seq, score in beams:
            if seq[0,-1].item() == EOS_IDX:
                completed.append((seq, score)); continue
            tmask = generate_square_subsequent_mask(seq.size(1), device)
            tgt_emb = model.pos_enc(model.tgt_tok(seq))
            tgt_proj = model.dec_proj(tgt_emb)
            out = model.decoder(tgt_proj, memory, tgt_mask=tmask, memory_key_padding_mask=mem_mask)
            logits = model.generator(out[:, -1, :])  # [1, V]
            log_probs = F.log_softmax(logits, dim=-1).squeeze(0)
            used = set(seq.flatten().tolist())
            penalized = log_probs.clone()
            for tid in used:
                if 0 <= tid < penalized.size(0):
                    penalized[tid] -= rep_pen
            topk = torch.topk(penalized, k=min(beam_size, penalized.size(0)))
            for tok_id, tok_score in zip(topk.indices.tolist(), topk.values.tolist()):
                new_seq = torch.cat([seq, torch.tensor([[tok_id]], device=device)], dim=1)
                new_score = score + float(tok_score)
                new_beams.append((new_seq, new_score))
        beams = sorted(new_beams, key=lambda x: x[1] / ((x[0].size(1) ** length_penalty)), reverse=True)[:beam_size]
        if len(completed) >= beam_size:
            break
    if len(completed) == 0:
        completed = beams
    best_seq, best_score = max(completed, key=lambda x: x[1] / (x[0].size(1) ** length_penalty))
    ids = [int(i) for i in best_seq.flatten().tolist() if i not in (BOS_IDX, EOS_IDX, PAD_IDX)]
    return char_decode(ids)


In [9]:
# Cell 8: instantiate model (tweak dims if needed), run a quick GPU forward test
# Optionally reduce dims for memory safety: uncomment to reduce
# TEXT_EMB_SIZE = 128
# GNN_HIDDEN_SIZE = 256

model = Seq2SeqTransformerGraphVideo(SRC_VOCAB_SIZE, TGT_VOCAB_SIZE,
                                     video_dim=VIDEO_EMB_SIZE, text_dim=TEXT_EMB_SIZE,
                                     hidden=GNN_HIDDEN_SIZE, nhead=4, num_dec_layers=2).to(DEVICE)
print("Model created. Params (M):", sum(p.numel() for p in model.parameters())/1e6)

# Quick forward test (no backward)
batch = next(iter(train_dl))
batch_gpu = move_batch_to_device(batch, DEVICE)
with torch.no_grad():
    logits, tgt_out = model(batch_gpu)
print("Forward OK:", logits.shape, tgt_out.shape)


Model created. Params (M): 80.234311
Forward OK: torch.Size([4, 28, 3655]) torch.Size([4, 28])


In [10]:
# Cell 9: save / load helpers
def save_checkpoint(path, model, optimizer=None, scheduler=None, step=None):
    ckpt = {"model_state_dict": model.state_dict()}
    if optimizer is not None:
        ckpt["optimizer_state_dict"] = optimizer.state_dict()
    if scheduler is not None:
        ckpt["scheduler_state_dict"] = scheduler.state_dict()
    if step is not None:
        ckpt["global_step"] = step
    torch.save(ckpt, path)
    print("Saved checkpoint to", path)

def load_checkpoint(path, model, optimizer=None, scheduler=None, map_location=DEVICE):
    ckpt = torch.load(path, map_location=map_location)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scheduler is not None and "scheduler_state_dict" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    gs = ckpt.get("global_step", 0)
    print("Loaded checkpoint", path, "global_step:", gs)
    return gs


In [11]:
# Cell 10: Stable training loop (autocast with device_type), sample printing uses beam
# Hyperparams
EPOCHS = 5
base_lr = 1e-4
warmup_steps = 500
grad_accum_steps = 1
log_every_steps = 100
sample_every = 500
save_every = 2000
checkpoint_path = "graph_vatex_char_checkpoint.pt"

opt = torch.optim.AdamW(model.parameters(), lr=base_lr, betas=(0.9,0.98), eps=1e-9)
def lr_lambda(step): return min((step+1)/warmup_steps, 1.0)
scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.0)

global_step = 0
start_time = time.time()
for epoch in range(1, EPOCHS+1):
    model.train()
    running_loss = 0.0
    for i, batch in enumerate(tqdm(train_dl, desc=f"Epoch {epoch}", leave=False)):
        batch = move_batch_to_device(batch, DEVICE)
        with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            logits, tgt_out = model(batch)
            logits = torch.nan_to_num(logits, nan=0.0, posinf=1e4, neginf=-1e4)
            B, T, V = logits.shape
            loss = criterion(logits.reshape(B*T, V), tgt_out.reshape(B*T))
            loss = loss / grad_accum_steps

        scaler.scale(loss).backward()
        if (i+1) % grad_accum_steps == 0:
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            scaler.step(opt)
            scaler.update()
            scheduler.step()
            opt.zero_grad(set_to_none=True)
            global_step += 1

        running_loss += float(loss.item()) * grad_accum_steps

        if global_step % log_every_steps == 0:
            avg_loss = running_loss / max(1, global_step)
            print(f"[epoch {epoch}] step {global_step} | loss {float(loss.item()):.4f} | avg {avg_loss:.4f} | lr {opt.param_groups[0]['lr']:.3e}")

        if global_step % sample_every == 0 and global_step>0:
            pair = val_pairs[global_step % len(val_pairs)]
            item = GraphVATEXDataset([pair])[0]
            pred = translate_beam(model, item, beam_size=4, max_len=80, rep_pen=3.0)
            print(f"\n[SAMPLE @ {global_step}] EN: {pair['en']}")
            print(f"   GT: {pair['zh']}")
            print(f"   PRED (beam): {pred}\n")

        if global_step % save_every == 0 and global_step>0:
            save_checkpoint(checkpoint_path, model, opt, scheduler, step=global_step)

    print(f"Epoch {epoch} ended. elapsed {time.time()-start_time:.0f}s, global_step={global_step}")
    # optional small validation BLEU here if you like


Epoch 5:  96%|█████████▌| 31145/32489 [59:01<02:24,  9.31it/s]

[epoch 5] step 161100 | loss 1.9579 | avg 0.3181 | lr 1.000e-04


Epoch 5:  96%|█████████▌| 31245/32489 [59:13<02:36,  7.95it/s]

[epoch 5] step 161200 | loss 1.8710 | avg 0.3189 | lr 1.000e-04


Epoch 5:  96%|█████████▋| 31345/32489 [59:24<02:20,  8.14it/s]

...

Epoch 5: 100%|█████████▉| 32345/32489 [1:01:15<00:17,  8.07it/s]

[epoch 5] step 162300 | loss 1.6994 | avg 0.3281 | lr 1.000e-04


Epoch 5: 100%|█████████▉| 32445/32489 [1:01:27<00:05,  8.63it/s]

[epoch 5] step 162400 | loss 1.7342 | avg 0.3289 | lr 1.000e-04


Epoch 5 ended. elapsed 18930s, global_step=162445


In [12]:
# Cell 11: BLEU evaluation (char-level) using beam decode
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

@torch.no_grad()
def evaluate_bleu_char(model, pairs, num_samples=200, beam_size=4):
    model.eval()
    preds, refs = [], []
    for i in range(min(num_samples, len(pairs))):
        item = GraphVATEXDataset([pairs[i]])[0]
        pred = translate_beam(model, item, beam_size=beam_size, max_len=80, rep_pen=3.0)
        preds.append(list(pred))
        refs.append([list(pairs[i]["zh"])])
    smoothie = SmoothingFunction().method4
    return corpus_bleu(refs, preds, smoothing_function=smoothie) * 100

print(evaluate_bleu_char(model, val_pairs, num_samples=100))


19.948738637756396


In [13]:
# Cell 12: small diagnostics - inspect batch / print top-k logits for a sample
@torch.no_grad()
def inspect_sample_logits(model, pair, topk=30):
    model.eval()
    item = GraphVATEXDataset([pair])[0]
    batch = collate_graph_vatex([item])
    batch = move_batch_to_device(batch, DEVICE)
    logits, tgt_out = model(batch)
    probs0 = F.softmax(logits[0,0,:], dim=-1)
    tk = torch.topk(probs0, k=topk)
    for idx, score in zip(tk.indices.tolist(), tk.values.tolist()):
        token = id2char[idx] if idx < TGT_VOCAB_SIZE else tok_src.convert_ids_to_tokens([idx])
        print(idx, token, score)

# Example usage:
inspect_sample_logits(model, val_pairs[0], topk=30)


95 一 0.9402170181274414
750 在 0.03660960868000984
1599 有 0.003850033972412348
117 两 0.003320750780403614
120 个 0.003259493736550212
181 人 0.0006747557199560106
1126 当 0.0006610091077163815
1270 房 0.00044978028745390475
3223 这 0.00041008362313732505
529 另 0.000231491620070301
1960 滑 0.00020166310423519462
1773 正 0.000188087418791838
2188 用 0.0001637040841160342
292 健 0.00014320238551590592
604 和 0.00013464814401231706
100 三 0.0001300708099734038
3569 骑 0.00012876254913862795
3462 雪 0.00012599455658346415
2750 舞 0.00011362974328221753
974 小 0.00011080153490183875
3445 随 0.00010992217721650377
1066 帮 0.00010409217793494463
945 室 0.00010222381388302892
195 他 0.00010173073678743094
517 双 0.0001011758649838157
857 女 9.802999556995928e-05
102 下 8.835587505018339e-05
369 几 8.214650733862072e-05
836 天 8.137574332067743e-05
2195 电 8.017037907848135e-05


In [14]:
# Save inference-only weights (good for deployment)
torch.save({"model_state_dict": model.state_dict(),
            "tgt_vocab": char_list,
            "src_tokenizer_name": SRC_TOK_NAME},
           "graph_vatex_char_inference_only.pt")
print("Saved inference-only checkpoint.")


Saved inference-only checkpoint.


In [16]:
# Inference loader example to run in a fresh script/notebook
ckpt = torch.load("graph_vatex_char_inference_only.pt", map_location=DEVICE)
model = Seq2SeqTransformerGraphVideo(SRC_VOCAB_SIZE, len(ckpt["tgt_vocab"]),
                                     video_dim=VIDEO_EMB_SIZE, text_dim=TEXT_EMB_SIZE,
                                     hidden=GNN_HIDDEN_SIZE, nhead=4, num_dec_layers=2).to(DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# translate function wrapper (uses beam)
def translate_text_with_video(model, en_text, video_feats_np, beam_size=4):
    # video_feats_np: numpy array [T, VIDEO_EMB_SIZE]
    pair = {"video_id": "tmp", "en": en_text, "zh": ""}  # create pair
    # Build dataset item manually
    item = {"video_id": "tmp",
            "en": en_text,
            "zh": ""}
    # manually build dataset item structure
    item_ds = {
        "video_id": "tmp",
        "src_tokens": torch.tensor(tok_src.encode(en_text, add_special_tokens=True, truncation=True, max_length=MAX_TOK_LEN), dtype=torch.long),
        "tgt_tokens": torch.tensor([BOS_IDX, EOS_IDX], dtype=torch.long),
        "video_features": torch.from_numpy(video_feats_np).float()
    }
    # use collate wrapper to get proper shapes
    from copy import deepcopy
    ds_item = GraphVATEXDataset([{"video_id":"tmp","en":en_text,"zh":""}])[0]  # simpler route if dataset available
    return translate_beam(model, ds_item, beam_size=beam_size, max_len=80, rep_pen=3.0)

translate_text_with_video(model, "A man rides a bike", np.zeros((1,VIDEO_EMB_SIZE)))


'一个男人骑着自行车，然后把他停下来。'

In [17]:
# BLEU (char-level) on 500 samples
print("BLEU (500):", evaluate_bleu_char(model, val_pairs, num_samples=500, beam_size=4))

BLEU (500): 21.092021704395727


In [26]:
# --- Cell A: Text-only Evaluation Mode (no video) ---
@torch.no_grad()
def translate_beam_text_only(model, item, beam_size=4, max_len=64, length_penalty=0.7, rep_pen=3.0):
    """
    Same as translate_beam but ignores all video features.
    """
    model.eval()
    batch = collate_graph_vatex([item])
    
    # Replace video features with zeros
    zero_feats = [torch.zeros_like(vf) for vf in batch["video_features_list"]]
    memory, mem_mask = model._encode_graph_batch(batch["src_tokens"], batch["src_padding_mask"], zero_feats)

    device = DEVICE
    beams = [(torch.tensor([[BOS_IDX]], device=device), 0.0)]
    completed = []
    for step in range(max_len):
        new_beams = []
        for seq, score in beams:
            if seq[0, -1].item() == EOS_IDX:
                completed.append((seq, score))
                continue
            tmask = generate_square_subsequent_mask(seq.size(1), device)
            tgt_emb = model.pos_enc(model.tgt_tok(seq))
            tgt_proj = model.dec_proj(tgt_emb)
            out = model.decoder(tgt_proj, memory, tgt_mask=tmask, memory_key_padding_mask=mem_mask)
            logits = model.generator(out[:, -1, :])
            log_probs = F.log_softmax(logits, dim=-1).squeeze(0)
            
            used = set(seq.flatten().tolist())
            penalized = log_probs.clone()
            for tid in used:
                if 0 <= tid < penalized.size(0):
                    penalized[tid] -= rep_pen
            
            topk = torch.topk(penalized, k=min(beam_size, penalized.size(0)))
            for tok_id, tok_score in zip(topk.indices.tolist(), topk.values.tolist()):
                new_seq = torch.cat([seq, torch.tensor([[tok_id]], device=device)], dim=1)
                new_score = score + float(tok_score)
                new_beams.append((new_seq, new_score))
        beams = sorted(new_beams, key=lambda x: x[1] / ((x[0].size(1) ** length_penalty)), reverse=True)[:beam_size]
        if len(completed) >= beam_size:
            break

    if len(completed) == 0:
        completed = beams
    best_seq, best_score = max(completed, key=lambda x: x[1] / (x[0].size(1) ** length_penalty))
    ids = [int(i) for i in best_seq.flatten().tolist() if i not in (BOS_IDX, EOS_IDX, PAD_IDX)]
    return char_decode(ids)


@torch.no_grad()
def evaluate_bleu_text_only(model, pairs, num_samples=200, beam_size=4):
    """
    Evaluate BLEU ignoring video features (text-only).
    """
    model.eval()
    preds, refs = [], []
    for i in range(min(num_samples, len(pairs))):
        item = GraphVATEXDataset([pairs[i]])[0]
        pred = translate_beam_text_only(model, item, beam_size=beam_size, max_len=80, rep_pen=3.0)
        preds.append(list(pred))
        refs.append([list(pairs[i]["zh"])])
    smoothie = SmoothingFunction().method4
    bleu = corpus_bleu(refs, preds, smoothing_function=smoothie) * 100
    return bleu


In [27]:
# --- Cell B: Run text-only BLEU evaluation ---
bleu_text_only_100 = evaluate_bleu_text_only(model, val_pairs, num_samples=100)
print(f"Text-only BLEU (100 samples): {bleu_text_only_100:.3f}")

bleu_text_only_500 = evaluate_bleu_text_only(model, val_pairs, num_samples=500)
print(f"Text-only BLEU (500 samples): {bleu_text_only_500:.3f}")


Text-only BLEU (100 samples): 18.994
Text-only BLEU (500 samples): 19.154


In [28]:
# --- Cell C: Compare text+video vs text-only for a sample ---
sample_idx = random.randint(0, len(val_pairs)-1)
pair = val_pairs[sample_idx]
item = GraphVATEXDataset([pair])[0]

pred_video = translate_beam(model, item, beam_size=4, max_len=80, rep_pen=3.0)
pred_text = translate_beam_text_only(model, item, beam_size=4, max_len=80, rep_pen=3.0)

print("EN:", pair['en'])
print("GT:", pair['zh'])
print("Video-guided:", pred_video)
print("Text-only   :", pred_text)


EN: Something is burnt in air by a lady and she is talking and laughing
GT: 一位女士吹灭了在烧烤物上的火，然后发出笑声。
Video-guided: 一个穿着红色衣服的女人正在吹气。
Text-only   : 在一个房间里面，一位女士正在不停的说话。
